In [34]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()
llm = ChatGroq(
    api_key=os.getenv("API_KEY"),
    model="openai/gpt-oss-120b"
)

In [35]:
from langchain_core.tools import tool
@tool
def get_wallet_info(email:str)->str:
    """GET THE WALLET INFORMATION OF THE USER OF MULTIWALLET""" 
    return "Primary Wallet"

@tool
def get_user_profile(email:str)->str:
    """GET THE BASIC INFORMATION OF THE USER OF MULTIWALLET """
    return """
        NAME : Priyanshu Kumar
        Status : Account Active
        """

In [36]:
LLM_with_tools=llm.bind_tools([get_user_profile,get_wallet_info])

In [37]:
from typing import TypedDict,Annotated
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages
class AgentState(TypedDict):
    messages:Annotated[list[AnyMessage],add_messages]

In [38]:
def llm_node(state:AgentState):
    response=LLM_with_tools.invoke(state["messages"])
    return {"messages":[response]}

In [39]:
from langgraph.prebuilt import ToolNode
tool_node=ToolNode([get_user_profile,get_wallet_info])

In [40]:
from langgraph.graph import StateGraph, START
from langgraph.prebuilt import tools_condition

graph = StateGraph(AgentState)

graph.add_node("llm", llm_node)
graph.add_node("tools", tool_node)

graph.add_edge(START, "llm")

graph.add_conditional_edges(
    "llm",
    tools_condition
)

graph.add_edge("tools", "llm")

app = graph.compile()

In [41]:
result = app.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is my wallet information? My email is test@gmail.com"
        }
    ]
})

print(result)

{'messages': [HumanMessage(content='What is my wallet information? My email is test@gmail.com', additional_kwargs={}, response_metadata={}, id='651ab85d-3a6e-4576-83ef-269088b4b391'), AIMessage(content='', additional_kwargs={'reasoning_content': "The user asks for wallet information. We need to retrieve wallet info using function get_wallet_info with email. We'll call function.", 'tool_calls': [{'id': 'fc_99dae9f6-da7e-4aa3-943d-5a22fe6e29c2', 'function': {'arguments': '{"email":"test@gmail.com"}', 'name': 'get_wallet_info'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 169, 'total_tokens': 225, 'completion_time': 0.118177127, 'completion_tokens_details': {'reasoning_tokens': 26}, 'prompt_time': 0.007642865, 'prompt_tokens_details': None, 'queue_time': 0.402344844, 'total_time': 0.125819992}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e5b4e54fbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logp